# Pratica 08 — Modelli di linguaggio n-gramma

**Corso:** NLP Classico con Python · **Lezione 08** · **Tempo stimato:** 60-75 minuti
**Notebook di lezione:** `notebooks_clase/clase08_modelos_lenguaje_ngramas_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una o più celle con `# TODO`: completale.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere.
- Gli esercizi da 1 a 5 usano un corpus di quattro frasi e cifre che puoi calcolare a mano. La sfida usa l'estratto del Chisciotte (in spagnolo) di `datos/gutenberg/` (in Colab, carica la cartella `datos/`): un testo letterario antico, non l'italiano di oggi (*esempio illustrativo*).
- Le **soluzioni** sono alla fine del notebook, in una sezione separata. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Librerie: solo `numpy` e la libreria standard.

---
## Cella 0 — Setup e dati

In [ ]:
import subprocess, sys

def asegurar(paquete, modulo):
    """Installa il pacchetto solo se manca (idempotente e silenzioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)

asegurar("numpy", "numpy")
asegurar("matplotlib", "matplotlib")
print("Librerie pronte.")

In [ ]:
import os
import re
import math
import random
from collections import Counter

import numpy as np

random.seed(42)                          # seme fisso
np.random.seed(42)

def cargar_datos(nombre):
    """Cerca un file di dati in datos/, ../datos/ e /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/14-nlp-classico/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "Non trovo " + nombre + ". Carica la cartella datos/ del corso (in Colab: pannello dei file) o lasciala accanto al notebook.")

INI, FIN, UNK = "<s>", "</s>", "<unk>"     # inizio di frase, fine di frase, fuori vocabolario
ENCAB = re.compile(r"^(Capítulo|CAPITOLO)\b")   # dato-es

def capitulos(ruta):
    """Legge un estratto e restituisce una lista di capitoli (liste di paragrafi)."""
    caps = []
    for parrafo in open(ruta, encoding="utf-8").read().split("\n\n"):
        parrafo = parrafo.strip()
        if not parrafo:
            continue
        if ENCAB.match(parrafo):
            caps.append([])
        else:
            caps[-1].append(parrafo)
    return caps

def oraciones(parrafos):
    """Divide in frasi (. ! ? …) e in token minuscoli (senza punteggiatura)."""
    salida = []
    for p in parrafos:
        for o in re.split(r"(?<=[.!?…])\s+", p):
            toks = re.findall(r"\w+'|\w+", o.lower())
            if toks:
                salida.append(toks)
    return salida

print("Funzioni ausiliarie caricate.")

In [ ]:
# datos-es
# Corpus minuscolo (4 frasi per lingua) da calcolare a mano
CHICO = {
    "spa": [["un", "gato", "come", "pescado"], ["un", "gato", "duerme"],
            ["un", "perro", "come", "carne"], ["un", "perro", "duerme"]],
    "ita": [["il", "gatto", "mangia", "pesce"], ["il", "gatto", "dorme"],
            ["il", "cane", "mangia", "carne"], ["il", "cane", "dorme"]],
}

# Conteggi di bigrammi del corpus minuscolo scritti a mano (spagnolo), per verificare senza dipendere da altri esercizi
BI_EJ = {("<s>", "un"): 4, ("un", "gato"): 2, ("un", "perro"): 2, ("gato", "come"): 1, ("gato", "duerme"): 1,
         ("perro", "come"): 1, ("perro", "duerme"): 1, ("come", "pescado"): 1, ("come", "carne"): 1,
         ("duerme", "</s>"): 2, ("pescado", "</s>"): 1, ("carne", "</s>"): 1}
CTX_EJ = {"<s>": 4, "un": 4, "gato": 2, "perro": 2, "come": 2, "duerme": 2, "pescado": 1, "carne": 1}
V_EJ = ["un", "gato", "perro", "come", "duerme", "pescado", "carne", FIN, UNK]   # |V| = 9

## Esercizio 1 — Contare bigrammi con `<s>` e `</s>`

Completa `contar_bigramas(oras)`. `oras` è una lista di frasi (liste di token). Ogni frase viene circondata da `<s>` all'inizio e `</s>` alla fine, e la funzione restituisce due `Counter`: i bigrammi `(v, w)` e i contesti `v` (quante volte `v` compare come parola precedente).

Prima di programmare: in `CHICO["spa"]` (o in quello italiano, con `il`), quanti bigrammi distinti ci sono e quanto vale il contesto `un`? (12 e 4.)

In [ ]:
def contar_bigramas(oras):
    bi, ctx = Counter(), Counter()
    for o in oras:
        seq = [INI] + o + [FIN]
        for v, w in zip(seq, seq[1:]):
            pass    # TODO: somma 1 a bi[(v, w)] e a ctx[v]
    return bi, ctx

# Controllo automatico
_bi, _ctx = contar_bigramas(CHICO["spa"])
assert isinstance(_bi, Counter) and len(_bi) > 0, "Completa le righe contrassegnate con TODO"
assert len(_bi) == 12 and _ctx["un"] == 4 and _bi[("un", "gato")] == 2, "rivedi i conteggi: ci sono 12 bigrammi distinti e c(un) = 4"
assert _bi[(INI, "un")] == 4 and _bi[("duerme", FIN)] == 2 and _ctx[INI] == 4, "non dimenticare <s> e </s>"
assert dict(_bi) == BI_EJ and dict(_ctx) == CTX_EJ
_bi2, _ = contar_bigramas(CHICO["ita"])
assert len(_bi2) == 12 and _bi2[("il", "gatto")] == 2
print("Esercizio 1 corretto ✔")

## Esercizio 2 — Probabilità del bigramma: MLE e Laplace

Completa `p_bigrama(w, v, bi, ctx, V, a=0.0)`: la probabilità `P(w | v)` con `(c(v, w) + a) / (c(v) + a·|V|)`. Con `a = 0` è lo stimatore per conteggio (MLE) e con `a = 1` è il lisciamento di Laplace. Attenzione al contesto `v` che non esiste in `ctx`: con `a = 0` la probabilità è `0.0` (nessuna divisione per zero).

Calcola prima a mano con `BI_EJ` (in spagnolo) e `|V| = 9`: `P(gato | un)` = 1/2; con Laplace, `P(pescado | un)` = 1/13 e `P(gato | un)` = 3/13.

In [ ]:
def p_bigrama(w, v, bi, ctx, V, a=0.0):
    numerador = None    # TODO: c(v, w) + a
    denominador = None  # TODO: c(v) + a·|V|
    return None         # TODO: numeratore / denominatore, o 0.0 se il denominatore è 0

# Controllo automatico
assert p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ) is not None, "Completa le righe contrassegnate con TODO"
assert p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ) == 0.5 and p_bigrama("pescado", "un", BI_EJ, CTX_EJ, V_EJ) == 0.0
assert np.isclose(p_bigrama("pescado", "un", BI_EJ, CTX_EJ, V_EJ, 1.0), 1 / 13) and np.isclose(p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ, 1.0), 3 / 13), "con Laplace: (0 + 1) / (4 + 9) e (2 + 1) / (4 + 9)"
assert p_bigrama("gato", "zzz", BI_EJ, CTX_EJ, V_EJ) == 0.0, "contesto sconosciuto con a = 0: 0.0"
for _v in ("un", "come", "zzz"):
    for _a in (0.5, 1.0):
        assert np.isclose(sum(p_bigrama(w, _v, BI_EJ, CTX_EJ, V_EJ, _a) for w in V_EJ), 1.0), "con a > 0 le probabilità condizionate sommano a 1"
assert np.isclose(sum(p_bigrama(w, "un", BI_EJ, CTX_EJ, V_EJ) for w in V_EJ), 1.0)
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Perplessità di un modello a bigrammi

Completa `perplejidad_bigrama(oras, prob)`. `prob(w, v)` restituisce la probabilità di `w` dato `v`. La perplessità è `exp(-(1/N) · Σ ln P)`, dove la somma percorre ogni token e il `</s>` finale di ogni frase (con `<s>` come contesto iniziale) e `N` conta quei token, **senza** contare `<s>`. Se una probabilità è 0, restituisce `float("inf")`.

A mano: con MLE su `CHICO`, la frase `un perro duerme` (in italiano `il cane dorme`) ha `P = 1/4` e `N = 4`: perplessità `√2 ≈ 1,414`. Un modello uniforme su 9 parole ha perplessità 9.

In [ ]:
def perplejidad_bigrama(oras, prob):
    logsum, N = 0.0, 0
    for o in oras:
        seq = [INI] + o + [FIN]
        for v, w in zip(seq, seq[1:]):
            p = prob(w, v)
            # TODO: se p <= 0 restituisci float("inf"); altrimenti somma math.log(p) a logsum
            N += 1
    return None      # TODO: math.exp(-logsum / N)

# Controllo automatico
_mle = lambda w, v: BI_EJ.get((v, w), 0) / CTX_EJ[v] if v in CTX_EJ else 0.0
_pp = perplejidad_bigrama([["un", "perro", "duerme"]], _mle)
assert _pp is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_pp, math.sqrt(2)), "P = 1 · 1/2 · 1/2 · 1 = 1/4 con N = 4 token (contando </s>): perplessità √2"
assert np.isclose(perplejidad_bigrama([["un", "perro", "duerme"], ["un", "gato", "come", "carne"]], lambda w, v: 1 / 9), 9.0), "un modello uniforme su 9 parole ha perplessità 9"
assert perplejidad_bigrama([["un", "pescado"]], _mle) == float("inf"), "un bigramma mai visto (probabilità 0) rende infinita la perplessità"
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Vocabolario e trattamento dell'OOV

Completa due funzioni: `construir_vocab(oras, minimo)` restituisce l'insieme delle parole con almeno `minimo` occorrenze **nelle frasi che le vengono passate** (l'addestramento), e `reemplazar_oov(oras, vocab)` restituisce le frasi con `<unk>` al posto di ogni parola che non sia in `vocab`. Regola: il vocabolario si costruisce **solo con l'addestramento** e si applica uguale al test.

Conta a mano con `[["a","b","a"],["a","c"],["b","d","a"]]`: `a` compare 4 volte, `b` 2, `c` e `d` 1. Con `minimo = 2` il vocabolario è `{a, b}`.

In [ ]:
def construir_vocab(oras, minimo):
    cuenta = Counter(t for o in oras for t in o)
    return None      # TODO: insieme delle parole con cuenta[t] >= minimo

def reemplazar_oov(oras, vocab):
    return None      # TODO: lista di frasi in cui ogni token fuori da vocab è sostituito con UNK

# Controllo automatico
_oras = [["a", "b", "a"], ["a", "c"], ["b", "d", "a"]]
_voc = construir_vocab(_oras, 2)
assert _voc is not None, "Completa le righe contrassegnate con TODO"
assert _voc == {"a", "b"} and construir_vocab(_oras, 1) == {"a", "b", "c", "d"} and construir_vocab(_oras, 5) == set()
assert reemplazar_oov(_oras, _voc) == [["a", "b", "a"], ["a", UNK], ["b", UNK, "a"]], "ogni parola fuori dal vocabolario passa a <unk>"
assert reemplazar_oov([["a", "z"]], _voc) == [["a", UNK]] and _oras[1] == ["a", "c"], "non modificare la lista originale"
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Dividere per capitoli contigui

Completa `dividir_por_capitulos(caps, n_train, n_dev)`. `caps` è una lista di capitoli (ognuno una lista di paragrafi). Restituisce `(train, dev, test)`, ciascuno come lista di frasi (usa `oraciones`): `train` sono i primi `n_train` capitoli, `dev` i successivi `n_dev` e `test` **tutti i restanti**. Senza mescolare: un modello reale si usa su testo che arriva dopo.

In [ ]:
def dividir_por_capitulos(caps, n_train, n_dev):
    # TODO: appiattisci con oraciones(...) i capitoli caps[:n_train], poi i successivi n_dev e poi il resto
    train = dev = test = None
    return train, dev, test

# Controllo automatico
_caps = [["a a. b b."], ["c c."], ["d d. e e.", "f f."], ["g g."], ["h h. i i."]]
_tr, _dv, _te = dividir_por_capitulos(_caps, 3, 1)
assert _tr is not None, "Completa le righe contrassegnate con TODO"
assert _tr == [["a", "a"], ["b", "b"], ["c", "c"], ["d", "d"], ["e", "e"], ["f", "f"]], "train: i primi tre capitoli, in ordine"
assert _dv == [["g", "g"]] and _te == [["h", "h"], ["i", "i"]], "dev: il capitolo successivo; test: il resto"
_, _, _te2 = dividir_por_capitulos(_caps, 2, 2)
assert _te2 == [["h", "h"], ["i", "i"]]
print("Esercizio 5 corretto ✔")

---
## Sfida finale — Interpolare bigramma e unigramma sul Chisciotte

Con le tue funzioni, addestra su `datos/gutenberg/quijote_cap01-14.txt` (o l'italiano, cambiando il percorso e la divisione dei capitoli): 10 capitoli di addestramento, 2 di sviluppo e 2 di test, `minimo = 2` per il vocabolario. Scrivi `p_interp(w, v, lam)`: `lam · P_bigramma(w | v) + (1 − lam) · P_unigramma(w)` (usa il bigramma MLE con `a = 0`; se il contesto non esiste, usa solo l'unigramma). Scegli `lam` tra `LAMBDAS` con la perplessità di **sviluppo** e calcola la perplessità di **test**. Deve essere finita e migliore di quella di Laplace `a = 1` del bigramma. Ricorda: è un corpus letterario di quattro secoli fa, non lo spagnolo di oggi.

In [ ]:
LAMBDAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
_ruta, _ntr, _ndv = "gutenberg/quijote_cap01-14.txt", 10, 2

caps = capitulos(cargar_datos(_ruta))
train, dev, test = dividir_por_capitulos(caps, _ntr, _ndv)
vocab = construir_vocab(train, 2)
train_u, dev_u, test_u = (reemplazar_oov(x, vocab) for x in (train, dev, test))
bi, ctx = contar_bigramas(train_u)
V = sorted(vocab | {FIN, UNK})
uni = Counter(t for o in train_u for t in o + [FIN])
total = sum(uni.values())

def p_interp(w, v, lam):
    p_uni = uni[w] / total
    if ctx.get(v, 0) == 0:
        return None       # TODO: contesto sconosciuto: solo l'unigramma
    return None           # TODO: lam · P_bigramma (MLE) + (1 − lam) · P_unigramma

# TODO: scegli lam_mejor tra LAMBDAS con la perplessità di sviluppo (dev_u) e calcola pp_test con test_u
lam_mejor, pp_test = None, None
pp_laplace = perplejidad_bigrama(test_u, lambda w, v: p_bigrama(w, v, bi, ctx, V, 1.0))

# Controllo automatico
assert lam_mejor in LAMBDAS and pp_test is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(sum(p_interp(w, "un" if "un" in ctx else next(iter(ctx)), lam_mejor) for w in V), 1.0), "l'interpolazione deve sommare 1 su V"
assert np.isclose(sum(p_interp(w, "zzz", lam_mejor) for w in V), 1.0), "e anche con un contesto sconosciuto"
assert all(perplejidad_bigrama(dev_u, lambda w, v: p_interp(w, v, lam_mejor)) <= perplejidad_bigrama(dev_u, lambda w, v, l=l: p_interp(w, v, l)) + 1e-9 for l in LAMBDAS), "lam_mejor deve minimizzare la perplessità di sviluppo"
assert np.isclose(pp_test, perplejidad_bigrama(test_u, lambda w, v: p_interp(w, v, lam_mejor)))
assert math.isfinite(pp_test) and pp_test < pp_laplace, "deve essere finita e migliore di Laplace a = 1"
print("Sfida finale corretta ✔")
print("lam =", lam_mejor, "| PP test =", round(pp_test, 1), "| PP Laplace =", round(pp_laplace, 1))

**Le tue conclusioni (scrivi 3-4 frasi):** perché il lisciamento di Laplace con `a = 1` peggiora tanto un modello con vocabolario grande e pochi dati? Perché non si possono confrontare direttamente la perplessità di un modello con `minimo = 2` e quella di un altro con `minimo = 20`, né quella di un testo in spagnolo con quella di uno in italiano?

*(La tua risposta qui.)*

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
def contar_bigramas(oras):
    bi, ctx = Counter(), Counter()
    for o in oras:
        seq = [INI] + o + [FIN]
        for v, w in zip(seq, seq[1:]):
            bi[(v, w)] += 1
            ctx[v] += 1
    return bi, ctx

# Controllo automatico
_bi, _ctx = contar_bigramas(CHICO["spa"])
assert isinstance(_bi, Counter) and len(_bi) > 0, "Completa le righe contrassegnate con TODO"
assert len(_bi) == 12 and _ctx["un"] == 4 and _bi[("un", "gato")] == 2, "rivedi i conteggi: ci sono 12 bigrammi distinti e c(un) = 4"
assert _bi[(INI, "un")] == 4 and _bi[("duerme", FIN)] == 2 and _ctx[INI] == 4, "non dimenticare <s> e </s>"
assert dict(_bi) == BI_EJ and dict(_ctx) == CTX_EJ
_bi2, _ = contar_bigramas(CHICO["ita"])
assert len(_bi2) == 12 and _bi2[("il", "gatto")] == 2
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
def p_bigrama(w, v, bi, ctx, V, a=0.0):
    numerador = bi.get((v, w), 0) + a
    denominador = ctx.get(v, 0) + a * len(V)
    return numerador / denominador if denominador else 0.0

# Controllo automatico
assert p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ) is not None, "Completa le righe contrassegnate con TODO"
assert p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ) == 0.5 and p_bigrama("pescado", "un", BI_EJ, CTX_EJ, V_EJ) == 0.0
assert np.isclose(p_bigrama("pescado", "un", BI_EJ, CTX_EJ, V_EJ, 1.0), 1 / 13) and np.isclose(p_bigrama("gato", "un", BI_EJ, CTX_EJ, V_EJ, 1.0), 3 / 13), "con Laplace: (0 + 1) / (4 + 9) e (2 + 1) / (4 + 9)"
assert p_bigrama("gato", "zzz", BI_EJ, CTX_EJ, V_EJ) == 0.0, "contesto sconosciuto con a = 0: 0.0"
for _v in ("un", "come", "zzz"):
    for _a in (0.5, 1.0):
        assert np.isclose(sum(p_bigrama(w, _v, BI_EJ, CTX_EJ, V_EJ, _a) for w in V_EJ), 1.0), "con a > 0 le probabilità condizionate sommano a 1"
assert np.isclose(sum(p_bigrama(w, "un", BI_EJ, CTX_EJ, V_EJ) for w in V_EJ), 1.0)
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
def perplejidad_bigrama(oras, prob):
    logsum, N = 0.0, 0
    for o in oras:
        seq = [INI] + o + [FIN]
        for v, w in zip(seq, seq[1:]):
            p = prob(w, v)
            if p <= 0:
                return float("inf")
            logsum += math.log(p)
            N += 1
    return math.exp(-logsum / N)

# Controllo automatico
_mle = lambda w, v: BI_EJ.get((v, w), 0) / CTX_EJ[v] if v in CTX_EJ else 0.0
_pp = perplejidad_bigrama([["un", "perro", "duerme"]], _mle)
assert _pp is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(_pp, math.sqrt(2)), "P = 1 · 1/2 · 1/2 · 1 = 1/4 con N = 4 token (contando </s>): perplessità √2"
assert np.isclose(perplejidad_bigrama([["un", "perro", "duerme"], ["un", "gato", "come", "carne"]], lambda w, v: 1 / 9), 9.0), "un modello uniforme su 9 parole ha perplessità 9"
assert perplejidad_bigrama([["un", "pescado"]], _mle) == float("inf"), "un bigramma mai visto (probabilità 0) rende infinita la perplessità"
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
def construir_vocab(oras, minimo):
    cuenta = Counter(t for o in oras for t in o)
    return {t for t, k in cuenta.items() if k >= minimo}

def reemplazar_oov(oras, vocab):
    return [[t if t in vocab else UNK for t in o] for o in oras]

# Controllo automatico
_oras = [["a", "b", "a"], ["a", "c"], ["b", "d", "a"]]
_voc = construir_vocab(_oras, 2)
assert _voc is not None, "Completa le righe contrassegnate con TODO"
assert _voc == {"a", "b"} and construir_vocab(_oras, 1) == {"a", "b", "c", "d"} and construir_vocab(_oras, 5) == set()
assert reemplazar_oov(_oras, _voc) == [["a", "b", "a"], ["a", UNK], ["b", UNK, "a"]], "ogni parola fuori dal vocabolario passa a <unk>"
assert reemplazar_oov([["a", "z"]], _voc) == [["a", UNK]] and _oras[1] == ["a", "c"], "non modificare la lista originale"
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
def dividir_por_capitulos(caps, n_train, n_dev):
    def aplanar(cs):
        return [o for cap in cs for o in oraciones(cap)]
    return aplanar(caps[:n_train]), aplanar(caps[n_train:n_train + n_dev]), aplanar(caps[n_train + n_dev:])

# Controllo automatico
_caps = [["a a. b b."], ["c c."], ["d d. e e.", "f f."], ["g g."], ["h h. i i."]]
_tr, _dv, _te = dividir_por_capitulos(_caps, 3, 1)
assert _tr is not None, "Completa le righe contrassegnate con TODO"
assert _tr == [["a", "a"], ["b", "b"], ["c", "c"], ["d", "d"], ["e", "e"], ["f", "f"]], "train: i primi tre capitoli, in ordine"
assert _dv == [["g", "g"]] and _te == [["h", "h"], ["i", "i"]], "dev: il capitolo successivo; test: il resto"
_, _, _te2 = dividir_por_capitulos(_caps, 2, 2)
assert _te2 == [["h", "h"], ["i", "i"]]
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
LAMBDAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
_ruta, _ntr, _ndv = "gutenberg/quijote_cap01-14.txt", 10, 2

caps = capitulos(cargar_datos(_ruta))
train, dev, test = dividir_por_capitulos(caps, _ntr, _ndv)
vocab = construir_vocab(train, 2)
train_u, dev_u, test_u = (reemplazar_oov(x, vocab) for x in (train, dev, test))
bi, ctx = contar_bigramas(train_u)
V = sorted(vocab | {FIN, UNK})
uni = Counter(t for o in train_u for t in o + [FIN])
total = sum(uni.values())

def p_interp(w, v, lam):
    p_uni = uni[w] / total
    if ctx.get(v, 0) == 0:
        return p_uni
    return lam * bi.get((v, w), 0) / ctx[v] + (1 - lam) * p_uni

lam_mejor = min(LAMBDAS, key=lambda l: perplejidad_bigrama(dev_u, lambda w, v: p_interp(w, v, l)))
pp_test = perplejidad_bigrama(test_u, lambda w, v: p_interp(w, v, lam_mejor))
pp_laplace = perplejidad_bigrama(test_u, lambda w, v: p_bigrama(w, v, bi, ctx, V, 1.0))

assert lam_mejor in LAMBDAS and pp_test is not None, "Completa le righe contrassegnate con TODO"
assert np.isclose(sum(p_interp(w, "un" if "un" in ctx else next(iter(ctx)), lam_mejor) for w in V), 1.0), "l'interpolazione deve sommare 1 su V"
assert np.isclose(sum(p_interp(w, "zzz", lam_mejor) for w in V), 1.0), "e anche con un contesto sconosciuto"
assert all(perplejidad_bigrama(dev_u, lambda w, v: p_interp(w, v, lam_mejor)) <= perplejidad_bigrama(dev_u, lambda w, v, l=l: p_interp(w, v, l)) + 1e-9 for l in LAMBDAS), "lam_mejor deve minimizzare la perplessità di sviluppo"
assert np.isclose(pp_test, perplejidad_bigrama(test_u, lambda w, v: p_interp(w, v, lam_mejor)))
assert math.isfinite(pp_test) and pp_test < pp_laplace, "deve essere finita e migliore di Laplace a = 1"
print("Sfida finale corretta ✔")
print("lam =", lam_mejor, "| PP test =", round(pp_test, 1), "| PP Laplace =", round(pp_laplace, 1))

---
## Autovalutazione della lezione

Rispondi con onestà (una o due frasi ciascuna):

1. Quale esercizio ti è costato di più e perché?
2. Quale concetto non ti è rimasto chiaro (regola della catena contro ipotesi di Markov, `<s>` e `</s>`, lisciamento, interpolazione, perplessità, OOV)?
3. Cosa cambieresti o ripeteresti la prossima volta?
4. Quanto tempo hai dedicato in totale?

*(Le tue risposte qui.)*